In [23]:
import sqlite3
import random

Eesmärk on tekitada näiteandmestik, mis koosneks 100 transaktsioonist, mis sisaldavad peaverbi eitust ja 100 transaktsioonist, milles peaverb ei ole eitusvormis. Selleks tekitatakse näiteandmebaasid eelnevalt filtreeritud andmebaasidest *v32_data_filtered_neg.db* ja *v32_data_filtered_no_neg.db*, transaktsioonid valitakse juhuslikult.

### *filtered_neg_100.db*

In [24]:
# creating new database where resulting tables will be stored
con = sqlite3.connect("filtered_neg_100.db")
cur = con.cursor()

In [25]:
# algne
cur.execute('ATTACH DATABASE "v32_data_filtered_neg.db" AS filtered_neg')

In [26]:
# ajutine head_id tabelite jaoks
cur.execute('ATTACH DATABASE "temp_head_ids.db" AS tmp')

In [5]:
cur.execute('SELECT id FROM filtered_neg.transaction_head')
all_ids = cur.fetchall()

In [6]:
random.shuffle(all_ids)

In [7]:
ids_100 = all_ids[:100]
ids_100 = [ids[0] for ids in ids_100]

In [8]:
cur.execute("""
    DROP TABLE IF EXISTS tmp.neg_head_ids
""")

cur.execute("""
    CREATE TABLE tmp.neg_head_ids (
        head_id INTEGER
    )
    """
    )
    
for i in range(len(ids_100)):
    cur.execute("""
    INSERT INTO tmp.neg_head_ids
    (head_id)
    VALUES
    (?)
    """, (str(ids_100[i]),))
    
    con.commit()

In [27]:
#cur.execute("""
#    DROP TABLE IF EXISTS transaction_head
#""")

cur.execute("""
CREATE TABLE transaction_head AS
SELECT
    id,
    sentence_id,
    loc,
    verb,
    verb_compound,
    form,
    deprel,
    feats
FROM
    filtered_neg.transaction_head AS tr_head
INNER JOIN
    tmp.neg_head_ids AS head_ids
ON
    tr_head.id=head_ids.head_id
""")

In [28]:
cur.execute("""
CREATE TABLE transaction_row AS
SELECT
    id,
    tr.head_id,
    loc,
    loc_rel,
    deprel,
    form,
    lemma,
    feats,
    parent_loc,
    pos
FROM
    filtered_neg.transaction_row AS tr
INNER JOIN
    tmp.neg_head_ids AS head_ids
ON
    tr.head_id=head_ids.head_id
""")

In [29]:
con.close()

### *filtered_no_neg.db*

In [30]:
# creating new database where resulting tables will be stored
con = sqlite3.connect("filtered_no_neg_100.db")
cur = con.cursor()

In [31]:
# algne
cur.execute('ATTACH DATABASE "v32_data_filtered_no_neg.db" AS filtered_no_neg')

In [32]:
# ajutine head_id tabelite jaoks
cur.execute('ATTACH DATABASE "temp_head_ids.db" AS tmp')

In [16]:
cur.execute('SELECT id FROM filtered_no_neg.transaction_head')
all_ids = cur.fetchall()

In [17]:
random.shuffle(all_ids)

In [18]:
ids_100 = all_ids[:100]
ids_100 = [ids[0] for ids in ids_100]

In [19]:
cur.execute("""
    DROP TABLE IF EXISTS tmp.no_neg_head_ids
""")

cur.execute("""
    CREATE TABLE tmp.no_neg_head_ids (
        head_id INTEGER
    )
    """
    )
    
for i in range(len(ids_100)):
    cur.execute("""
    INSERT INTO tmp.no_neg_head_ids
    (head_id)
    VALUES
    (?)
    """, (str(ids_100[i]),))
    
    con.commit()

In [33]:
cur.execute("""
CREATE TABLE transaction_head AS
SELECT
    id,
    sentence_id,
    loc,
    verb,
    verb_compound,
    form,
    deprel,
    feats
FROM
    filtered_no_neg.transaction_head AS tr_head
INNER JOIN
    tmp.no_neg_head_ids AS head_ids
ON
    tr_head.id=head_ids.head_id
""")

In [34]:
cur.execute("""
CREATE TABLE transaction_row AS
SELECT
    id,
    tr.head_id,
    loc,
    loc_rel,
    deprel,
    form,
    lemma,
    feats,
    parent_loc,
    pos
FROM
    filtered_no_neg.transaction_row AS tr
INNER JOIN
    tmp.no_neg_head_ids AS head_ids
ON
    tr.head_id=head_ids.head_id
""")

In [35]:
con.close()

NB! Andmebaasist *filtered_no_neg.db* on näha, et sisse on jäänud mõned eitusvormis verbid.